# 01 · From an array to a scientific cube

## Context

A raster workflow has returned a three-dimensional array of observed daily
maximum temperatures. The values are real, but the array alone does not retain
which axis represents time, latitude, or longitude.

## Question

How can we reconstruct a self-describing cube and verify it as a map, a site
history, and an interactive space-time object?

## Analysis story

We will deliberately separate values from metadata, rebuild their scientific
context, and send the result through one minimal plotting pipe.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

## Prepare · Recover coordinates and provenance for the array

In [ ]:
from pathlib import Path

import xarray as xr

# Find the repository from either a root-level documentation build or a kernel
# started beside this notebook, then open the checksum-controlled PRISM extract.
data_path = next(
    candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc").exists()
)
prism = xr.open_dataset(data_path, engine="scipy").load()

# These assertions are part of the teaching contract: official source,
# canonical cube dimensions, complete daily time, and declared Celsius units.
assert prism.attrs["source"] == "PRISM Group, Oregon State University"
assert prism.attrs["is_synthetic"] == 0
assert prism.sizes == {"time": 30, "y": 24, "x": 24}
assert prism["tmax"].attrs["units"] == "degC"

import numpy as np

# Use an inspectable 18-day, 5-row × 6-column portion of the official grid.
source = prism["tmax"].isel(time=slice(0, 18), y=slice(8, 13), x=slice(8, 14))
values = source.values

# Reattach the coordinate and provenance fields that an anonymous NumPy array
# cannot carry. No temperatures are generated or altered in this conversion.
cube = xr.DataArray(
    values,
    dims=("time", "y", "x"),
    coords={name: source[name] for name in ("time", "y", "x")},
    name="tmax",
    attrs=dict(source.attrs),
)
cube.attrs.update(source=prism.attrs["source"], is_synthetic=0)

np.testing.assert_array_equal(cube.values, source.values)
assert cube.dims == ("time", "y", "x")
cube

## Figure 1 · Read the cube in familiar views

Compare a map from the cold outbreak with the history of one grid cell. Both
views must retain the observed PRISM values.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
cube.isel(time=15).plot(ax=axes[0], cmap="magma", cbar_kwargs={"label": "°C"})
axes[0].set_title("PRISM maximum temperature · 16 January")
cube.isel(y=2, x=3).plot(ax=axes[1], marker="o", color="#2f6f6d")
axes[1].set_title("One PRISM grid cell through time")
axes[1].set_ylabel("Daily maximum temperature (°C)")
plt.show()

## Pipe · Inspect the same evidence as a cube

The method remains one short sentence. The viewer is generated from the same
validated `DataArray`; it does not become a second data authority.

In [ ]:
from html import escape
from IPython.display import HTML
from cubedynamics import pipe, verbs as v

viewer = (
    pipe(cube)
    | v.plot(
        title="PRISM daily maximum temperature · Boulder region",
        cmap="magma",
        thin_time_factor=1,
    )
).unwrap()

# Isolate the complete viewer document from the surrounding MkDocs page.
viewer_srcdoc = escape(viewer.to_html(), quote=True)
HTML(
    f'''<iframe
        title="Interactive PRISM maximum-temperature cube"
        srcdoc="{viewer_srcdoc}"
        style="width: 100%; height: 760px; border: 1px solid #b8c5c2; border-radius: 4px;"
        sandbox="allow-scripts"
        loading="lazy"
    ></iframe>'''
)

## What the figure tells us

The mid-January cold outbreak is visible through depth and across the map. The
validation suite independently decodes all six HTML textures and checks them
against these source indices, including the declared back-face reversal.

## Try the next variation

Choose a different observed time window or spatial subset. The reconstruction
and pipe stay unchanged.